In [1]:
#Load and check monthly stays are flat
import duckdb
import pandas as pd

cal = duckdb.sql(
    "SELECT * FROM read_parquet('../data/gold/nyc/price_calendar.parquet')"
).df()

print("Monthly-stay adjustment range:",
      cal.loc[cal["stay_type"] == "monthly stay", "price_adjustment_pct"].min(),
      "to",
      cal.loc[cal["stay_type"] == "monthly stay", "price_adjustment_pct"].max())

print("Rows:", len(cal), " Listings:", cal["listing_id"].nunique())

Monthly-stay adjustment range: 0.0 to 0.0
Rows: 1936260  Listings: 21514


In [2]:
#Spot-check known high-demand dates for a sample listing
sample_id = cal.loc[cal["stay_type"] == "short stay", "listing_id"].iloc[0]
one_listing = cal[cal["listing_id"] == sample_id].sort_values("calendar_date")

print(one_listing[["calendar_date", "days_from_start", "base_price",
                    "price_adjustment_pct", "suggested_price",
                    "used_date_specific_excess"]].head(30).to_string(index=False))

# Check July 18 and July 4 specifically, if in this listing's window
print()
print(one_listing[one_listing["calendar_date"].isin(
    pd.to_datetime(["2026-07-18", "2026-07-04"])
)])

calendar_date  days_from_start  base_price  price_adjustment_pct  suggested_price  used_date_specific_excess
   2026-06-23                8       195.5               -0.0840           179.08                       True
   2026-06-24                9       195.5               -0.0510           185.53                       True
   2026-06-25               10       195.5                0.0270           200.78                       True
   2026-06-26               11       195.5                0.1500           224.83                       True
   2026-06-27               12       195.5                0.1500           224.83                       True
   2026-06-28               13       195.5                0.0150           198.43                       True
   2026-06-29               14       195.5               -0.1080           174.39                       True
   2026-06-30               15       195.5               -0.1290           170.28                       True
   2026-07-01      

In [3]:
#Overall distribution of adjustments
short = cal[cal["stay_type"] == "short stay"]
print(short["price_adjustment_pct"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).round(3))

# How often did we use the real date-level excess vs the weekday fallback?
print()
print(short["used_date_specific_excess"].value_counts(normalize=True).round(3) * 100)

count    449370.000
mean         -0.003
std           0.075
min          -0.140
5%           -0.123
25%          -0.051
50%          -0.018
75%           0.053
95%           0.150
max           0.150
Name: price_adjustment_pct, dtype: float64

used_date_specific_excess
True    100.0
Name: proportion, dtype: Float64


In [4]:
# Does every date in the 90-day window have a row in date_adjustment.parquet?
date_range = pd.date_range("2026-06-23", periods=90)
date_adj = duckdb.sql(
    "SELECT DISTINCT calendar_date FROM read_parquet('../data/gold/nyc/date_adjustment.parquet')"
).df()["calendar_date"]

missing = set(date_range.date) - set(date_adj.dt.date if hasattr(date_adj, 'dt') else date_adj)
print(f"Dates in the 90-day window missing from date_adjustment: {len(missing)}")
print(sorted(missing)[:10])

Dates in the 90-day window missing from date_adjustment: 0
[]


In [6]:
# How many nights hit each cap exactly?
at_max = (short["price_adjustment_pct"] == 0.15).mean() * 100
at_min = (short["price_adjustment_pct"] == -0.14).mean() * 100  # min observed was -0.140, check exact floor too
at_min_exact = (short["price_adjustment_pct"] <= -0.1499).mean() * 100

print(f"Nights at the +15% cap: {at_max:.1f}%")
print(f"Nights at or near the -15% floor: {at_min_exact:.1f}%")

# What does the underlying excess_pct distribution look like, before capping?
raw_excess = duckdb.sql("""
    SELECT excess_pct FROM read_parquet('../data/gold/nyc/date_adjustment.parquet')
""").df()
print()
print(raw_excess["excess_pct"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).round(2))

Nights at the +15% cap: 5.6%
Nights at or near the -15% floor: 0.0%

count    189.00
mean      -0.26
std        4.58
min      -10.50
5%        -7.78
25%       -2.90
50%       -1.00
75%        2.10
95%        8.22
max       16.60
Name: excess_pct, dtype: float64
